# Publish Regular 2D Grid from Image

This notebook shows how you can sign in and publish a Regular 2D Grid geoscience object from an image file (JPEG, PNG, TIFF, BMP, GIF, etc.) to your chosen Evo workspace.

**Important:** This notebook requires Python 3.10, 3.11, or 3.12 (not 3.14+) due to evo.notebooks dependencies.

In the second code cell we create a ServiceManagerWidget which will open a browser window and ask you to sign in.

Once signed in, a widget will be displayed to allow you to select your organisation and an Evo workspace.

__Required:__ In second code cell, replace `"your-client-id"` with your Evo app client ID before running the cell.

In [1]:
# Install local package into this notebook kernel (run once per environment)
%pip install -e ../..

Defaulting to user installation because normal site-packages is not writeableNote: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip



Obtaining file:///C:/Users/Maitri.Anvekar/Documents/evo_data_converters_PR/evo-data-converters/packages/image
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Checking if build backend supports build_editable: started
  Checking if build backend supports build_editable: finished with status 'done'
  Getting requirements to build editable: started
  Getting requirements to build editable: finished with status 'done'
  Installing backend dependencies: started
  Installing backend dependencies: finished with status 'done'
  Preparing editable metadata (pyproject.toml): started
  Preparing editable metadata (pyproject.toml): finished with status 'done'
  Building editable for evo-data-converters-image (pyproject.toml): started
  Building editable for evo-data-converters-image (pyproject.toml): finished with status 'done'
  Created wheel for evo-data-converters-image: filename=evo_data_converters_image-0.1.5-py3-none-any.whl size=6651 

In [ ]:
from evo.notebooks import ServiceManagerWidget

manager = await ServiceManagerWidget.with_auth_code(client_id="your-client-id").login()

C:\Users\Maitri.Anvekar\AppData\Roaming\Python\Python310\site-packages\evo\notebooks\authorizer.py:108: UserWarning: The evo.notebooks.AuthorizationCodeAuthorizer is not secure, and should only ever be used in Jupyter notebooks in a private environment.
  warnings.warn(


ServiceManagerWidget(children=(VBox(children=(HBox(children=(Image(value=b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIHDR…

## Convert and Publish Image to Regular 2D Grid

In the cell below we:
1. Specify the path to our image file (JPEG, PNG, TIFF, etc.)
2. Set the grid parameters (`origin`, `cell_size`, `coordinate_reference_system`)
3. Add optional tags
4. Call `convert_image_to_grid` to convert and publish the image

The image will be:
- Preserved as color for RGB/RGBA inputs
- Converted to grayscale only for grayscale inputs
- Pixel values extracted and stored in a parquet file
- Formatted according to the regular-2d-grid schema
- Published to your Evo workspace

**Note:** If you don't have workspaces available, use `evo_workspace_metadata` instead of `service_manager_widget`.

In [ ]:
from evo.data_converters.image import convert_image_to_grid
from pathlib import Path

# Path to your image file (JPEG, PNG, TIFF, etc.)
# Using relative path from notebook location
image_file = "data/input/sample_gradient.jpg"

# Grid origin in world coordinates [x, y, z]
origin = [572565.0, 6839415.0, 1000.0]

# Cell size [cell_size_x, cell_size_y]
cell_size = [30.0, 30.0]

# Coordinate Reference System (optional)
crs = {"epsg_code": 32632}  # WGS 84 / UTM zone 32N

# Tags to add to the geoscience object
tags = {"Source": "Jupyter Notebook", "Type": "Image"}

# Name for the grid (optional, defaults to filename)
grid_name = f"Image - {Path(image_file).stem}"

# Description (optional)
description = "A 2D grid generated from image file"

# Upload path in Evo workspace
upload_path = "grids/image_imports"

# Get selected workspace from the widget
selected_workspace = manager._service_manager.get_current_workspace()

print("Converting and publishing image to Evo workspace...")
print(f"Image: {Path(image_file).name}")
print(f"Workspace: {selected_workspace}")
print(f"Origin: {origin}")
print(f"Cell size: {cell_size}")

# Convert and publish
results = convert_image_to_grid(
    image_path=image_file,
    origin=origin,
    cell_size=cell_size,
    coordinate_reference_system=crs,
    tags=tags,
    name=grid_name,
    description=description,
    service_manager_widget=manager,
    upload_path=upload_path,
    publish_objects=True,
    overwrite_existing_objects=True,
)

# Print results
print("\n✅ Successfully published!")
print("\nPublished objects:")
for obj_metadata in results:
    print(f"  - {obj_metadata.name}")
    print(f"    ID: {obj_metadata.id}")

Converting and publishing image to Evo workspace...
Image: sample_gradient.jpg
Workspace: maitestmcp (ID: 1850bf93-ea96-4a91-945f-10aa73527d88)
Origin: [572565.0, 6839415.0, 1000.0]
Cell size: [30.0, 30.0]

✅ Successfully published!

Published objects:
  - Image - sample_gradient.json
    ID: 72fd4492-97d9-4db3-9171-f6cfff3e2a8a


## Simple Example with Defaults

If you don't need custom parameters, you can use a simpler call with default values:

In [4]:
from evo.data_converters.image import convert_image_to_grid

# Minimal example with defaults:
# - origin: [0, 0, 0]
# - cell_size: [1, 1]
# - no CRS
# - name derived from filename

results = convert_image_to_grid(
    image_path="data/input/sample_gradient.jpg",
    service_manager_widget=manager,
    upload_path="grids",
    tags={"Source": "Jupyter Notebook"},
)

print(f"Published: {results[0].name}")

Published: sample_gradient.json


## Convert Without Publishing (for testing)

You can also convert the JPEG without publishing to inspect the resulting object:

In [5]:
from evo.data_converters.image import convert_image_to_grid
import json

# Convert but don't publish
grid_objects = convert_image_to_grid(
    image_path="data/input/sample_gradient.jpg",
    service_manager_widget=manager,
    publish_objects=False,  # Don't publish
)

# Inspect the grid object
grid = grid_objects[0]
print(f"Grid name: {grid.name}")
print(f"Grid size: {grid.size}")
print(f"Cell size: {grid.cell_size}")
print(f"Origin: {grid.origin}")
print(f"Number of cell attributes: {len(grid.cell_attributes)}")

grid_dict = grid.as_dict()  # Use as_dict() instead of to_dict()
print("\nJSON representation:")
print(json.dumps(grid_dict, indent=2))

Grid name: sample_gradient
Grid size: [128, 106]
Cell size: [1.0, 1.0]
Origin: [0.0, 0.0, 0.0]
Number of cell attributes: 1

JSON representation:
{
  "name": "sample_gradient",
  "uuid": null,
  "description": "A 2D grid from image",
  "bounding_box": {
    "min_x": 0.0,
    "max_x": 128.0,
    "min_y": 0.0,
    "max_y": 106.0,
    "min_z": 0.0,
    "max_z": 0.0
  },
  "coordinate_reference_system": {
    "epsg_code": 4326
  },
  "origin": [
    0.0,
    0.0,
    0.0
  ],
  "size": [
    128,
    106
  ],
  "cell_size": [
    1.0,
    1.0
  ],
  "schema": "/objects/regular-2d-grid/1.3.0/regular-2d-grid.schema.json",
  "rotation": {
    "dip_azimuth": 0.0,
    "dip": 0.0,
    "pitch": 0.0
  },
  "cell_attributes": [
    {
      "name": "2d-grid-data-continuous",
      "key": "1a12f275d87091254e9ccd6ca71957536832e3f9e7aca8e40353beb7845a870e",
      "attribute_type": "scalar",
      "attribute_description": {
        "discipline": "Imagery",
        "type": "Grayscale Intensity"
      },
